# 1-10 Jan data issue with 5km RAL3

* Data issue with these dates - have to reprocess healpix for them (9/5/2025)
* z10 is simple - just move the affected donefiles
* z9-0 is more involved. Map the dates to time indices, then figure out which coarsening jobs to redo and shift these donefiles.
* Need to do 2D and 3D.
* All files renamed by 2025-05-09 12:00 - wait for z10 regridding then redo coarsening.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
mv_dates = pd.date_range('2021-01-01', '2021-01-11', freq='12h')

In [3]:
mv_dates

DatetimeIndex(['2021-01-01 00:00:00', '2021-01-01 12:00:00',
               '2021-01-02 00:00:00', '2021-01-02 12:00:00',
               '2021-01-03 00:00:00', '2021-01-03 12:00:00',
               '2021-01-04 00:00:00', '2021-01-04 12:00:00',
               '2021-01-05 00:00:00', '2021-01-05 12:00:00',
               '2021-01-06 00:00:00', '2021-01-06 12:00:00',
               '2021-01-07 00:00:00', '2021-01-07 12:00:00',
               '2021-01-08 00:00:00', '2021-01-08 12:00:00',
               '2021-01-09 00:00:00', '2021-01-09 12:00:00',
               '2021-01-10 00:00:00', '2021-01-10 12:00:00',
               '2021-01-11 00:00:00'],
              dtype='datetime64[ns]', freq='12h')

In [4]:
basedir = Path('/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5')
mv_paths = [basedir / f'regrid_{d}.done' for d in mv_dates]

In [5]:
all([d.exists() for d in mv_paths])

True

In [9]:
for p in mv_paths[:-1]:
    newpath = p.with_suffix('.jan_data_issue')
    print(f'{p} -> {newpath}')
    p.rename(newpath)

/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/regrid_2021-01-01 00:00:00.done -> /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/regrid_2021-01-01 00:00:00.jan_data_issue
/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/regrid_2021-01-01 12:00:00.done -> /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/regrid_2021-01-01 12:00:00.jan_data_issue
/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/regrid_2021-01-02 00:00:00.done -> /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/regrid_2021-01-02 00:00:00.jan_data_issue
/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/regrid_2021-01-02 12:00:00.done -> /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/regrid_2021-01-02 12:00:00.jan_data_issue
/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/regrid_2021-01-03 00:00:00.done -> /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/regrid_2021-01-03 00:00:00.jan_data_issue
/gws/

## Easy bit done. Now I have to figure out how these dates mape onto each of the coarsening steps.

* Idea is to map from times (1-10 Jan) to job indices at each lower zoom level.

In [10]:
import intake

In [12]:
cat = intake.open_catalog('https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml')['UK']

In [13]:
ds = cat['um_glm_n2560_RAL3p3'](zoom=10, time='PT1H').to_dask()

/home/users/mmuetz/miniforge3/envs/hackathon_env/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
/home/users/mmuetz/miniforge3/envs/hackathon_env/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


In [17]:
ds.sel(time=slice('2021-01-01 00:00', '2021-01-11 01:00')).time

<xarray.DataArray 'time' (time: 242)> Size: 2kB
array(['2021-01-01T00:00:00.000000000', '2021-01-01T01:00:00.000000000',
       '2021-01-01T02:00:00.000000000', ..., '2021-01-10T23:00:00.000000000',
       '2021-01-11T00:00:00.000000000', '2021-01-11T01:00:00.000000000'],
      dtype='datetime64[ns]')
Coordinates:
    crs      float64 8B ...
  * time     (time) datetime64[ns] 2kB 2021-01-01 ... 2021-01-11T01:00:00

In [33]:
time = ds.time.values
tmask = (time >= pd.Timestamp('2021-01-01 00:00')) & (time <= pd.Timestamp('2021-01-11 01:00'))

In [22]:
tmask.sum()

np.int64(242)

In [26]:
import numpy as np
affected_indices = np.where(tmask)[0]
affected_indices

array([8328, 8329, 8330, 8331, 8332, 8333, 8334, 8335, 8336, 8337, 8338,
       8339, 8340, 8341, 8342, 8343, 8344, 8345, 8346, 8347, 8348, 8349,
       8350, 8351, 8352, 8353, 8354, 8355, 8356, 8357, 8358, 8359, 8360,
       8361, 8362, 8363, 8364, 8365, 8366, 8367, 8368, 8369, 8370, 8371,
       8372, 8373, 8374, 8375, 8376, 8377, 8378, 8379, 8380, 8381, 8382,
       8383, 8384, 8385, 8386, 8387, 8388, 8389, 8390, 8391, 8392, 8393,
       8394, 8395, 8396, 8397, 8398, 8399, 8400, 8401, 8402, 8403, 8404,
       8405, 8406, 8407, 8408, 8409, 8410, 8411, 8412, 8413, 8414, 8415,
       8416, 8417, 8418, 8419, 8420, 8421, 8422, 8423, 8424, 8425, 8426,
       8427, 8428, 8429, 8430, 8431, 8432, 8433, 8434, 8435, 8436, 8437,
       8438, 8439, 8440, 8441, 8442, 8443, 8444, 8445, 8446, 8447, 8448,
       8449, 8450, 8451, 8452, 8453, 8454, 8455, 8456, 8457, 8458, 8459,
       8460, 8461, 8462, 8463, 8464, 8465, 8466, 8467, 8468, 8469, 8470,
       8471, 8472, 8473, 8474, 8475, 8476, 8477, 84

In [28]:
import sys
sys.path.insert(0, '/home/users/mmuetz/deploy/wcrp_hackathon/scripts/process_um_data')
from processing_config import processing_config

In [29]:
config = processing_config['glm.n2560_RAL3p3']

In [30]:
config['groups']['2d']['chunks']

{10: (1, 1048576),
 9: (4, 262144),
 8: (16, 65536),
 7: (64, 16384),
 6: (64, 16384),
 5: (64, 12288),
 4: (256, 3072),
 3: (1024, 768),
 2: (4096, 192),
 1: (16384, 48),
 0: (65536, 12)}

In [39]:
c9chunk = config['groups']['2d']['chunks'][9][0]

In [40]:
c9chunk

4

In [35]:
tmask.shape

(10489,)

In [51]:
nchunk = int(np.ceil(len(tmask) / c9chunk))

In [52]:
nchunk

2623

In [41]:
c9time = np.arange(nchunk).repeat(c9chunk)

In [42]:
c9time

array([   0,    0,    0, ..., 2622, 2622, 2622])

In [45]:
c9idx = np.unique(c9time[affected_indices])

In [46]:
c9idx

array([2082, 2083, 2084, 2085, 2086, 2087, 2088, 2089, 2090, 2091, 2092,
       2093, 2094, 2095, 2096, 2097, 2098, 2099, 2100, 2101, 2102, 2103,
       2104, 2105, 2106, 2107, 2108, 2109, 2110, 2111, 2112, 2113, 2114,
       2115, 2116, 2117, 2118, 2119, 2120, 2121, 2122, 2123, 2124, 2125,
       2126, 2127, 2128, 2129, 2130, 2131, 2132, 2133, 2134, 2135, 2136,
       2137, 2138, 2139, 2140, 2141, 2142])

In [48]:
Path('/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z9/2082.done').read_text().split('\n')[0]

'2025-05-07 14:02:55.840 | DEBUG    | __main__:coarsen_healpix_region:452 - (8328, 8332)'

In [49]:
Path('/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z9/2142.done').read_text().split('\n')[0]

'2025-05-07 14:03:21.343 | DEBUG    | __main__:coarsen_healpix_region:452 - (8568, 8572)'

In [50]:
affected_indices[[0, -1]]

array([8328, 8569])

**Conclusion:** `c9idx` contains the correct z9 files. Wrap up into func and do for all zooms.

## Move affected 2D donefiles

In [57]:
z_affected_idx = {}
for z in range(9, -1, -1):
    print(z)
    zchunk = config['groups']['2d']['chunks'][z][0]
    nchunk = int(np.ceil(len(tmask) / zchunk))
    ztime_idx = np.arange(nchunk).repeat(zchunk)
    z_affected_idx[z] = np.unique(ztime_idx[affected_indices])

9
8
7
6
5
4
3
2
1
0


In [58]:
z_affected_idx

{9: array([2082, 2083, 2084, 2085, 2086, 2087, 2088, 2089, 2090, 2091, 2092,
        2093, 2094, 2095, 2096, 2097, 2098, 2099, 2100, 2101, 2102, 2103,
        2104, 2105, 2106, 2107, 2108, 2109, 2110, 2111, 2112, 2113, 2114,
        2115, 2116, 2117, 2118, 2119, 2120, 2121, 2122, 2123, 2124, 2125,
        2126, 2127, 2128, 2129, 2130, 2131, 2132, 2133, 2134, 2135, 2136,
        2137, 2138, 2139, 2140, 2141, 2142]),
 8: array([520, 521, 522, 523, 524, 525, 526, 527, 528, 529, 530, 531, 532,
        533, 534, 535]),
 7: array([130, 131, 132, 133]),
 6: array([130, 131, 132, 133]),
 5: array([130, 131, 132, 133]),
 4: array([32, 33]),
 3: array([8]),
 2: array([2]),
 1: array([0]),
 0: array([0])}

In [ ]:
for z in range(9, -1, -1):
    z_idx = z_affected_idx[z]
    print(affected_indices[[0, -1]])
    print(f'Confirming idx for {z}')
    print(z_idx[0])
    print(Path(f'/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z{z}/{z_idx[0]}.done').read_text().split('\n')[0])
    print(z_idx[-1])
    print(Path(f'/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z{z}/{z_idx[-1]}.done').read_text().split('\n')[0])    

**Conclusion:** correctly generating index of file at each lower zoom level (affected indices 8328-8569 are always contained in the first and last at each zoom)

In [65]:
for z in range(9, -1, -1):
    z_idx = z_affected_idx[z]
    print(f'zoom {z}')
    for idx in z_idx:
        path = Path(f'/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z{z}/{idx}.done')
        #assert path.exists()
        newpath = path.with_suffix('.jan_data_issue')
        print(f'  {path} -> {newpath.name}')
        #newpath = path.rename()


zoom 9
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z9/2082.done -> 2082.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z9/2083.done -> 2083.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z9/2084.done -> 2084.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z9/2085.done -> 2085.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z9/2086.done -> 2086.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z9/2087.done -> 2087.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z9/2088.done -> 2088.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z9/2089.done -> 2089.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z9/2090.done -> 2090.jan_data_issue
  /

In [67]:
for z in range(9, -1, -1):
    z_idx = z_affected_idx[z]
    print(affected_indices[[0, -1]])
    print(f'Confirming idx for {z}')
    print(z_idx[0])
    print(Path(f'/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z{z}/{z_idx[0]}.jan_data_issue').read_text().split('\n')[0])
    print(z_idx[-1])
    print(Path(f'/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/2d/z{z}/{z_idx[-1]}.jan_data_issue').read_text().split('\n')[0])    

[8328 8569]
Confirming idx for 9
2082
2025-05-07 14:02:55.840 | DEBUG    | __main__:coarsen_healpix_region:452 - (8328, 8332)
2142
2025-05-07 14:03:21.343 | DEBUG    | __main__:coarsen_healpix_region:452 - (8568, 8572)
[8328 8569]
Confirming idx for 8
520
2025-05-07 14:47:45.387 | DEBUG    | __main__:coarsen_healpix_region:452 - (8320, 8336)
535
2025-05-07 14:48:04.352 | DEBUG    | __main__:coarsen_healpix_region:452 - (8560, 8576)
[8328 8569]
Confirming idx for 7
130
2025-05-07 15:03:02.550 | DEBUG    | __main__:coarsen_healpix_region:452 - (8320, 8384)
133
2025-05-07 15:04:39.321 | DEBUG    | __main__:coarsen_healpix_region:452 - (8512, 8576)
[8328 8569]
Confirming idx for 6
130
2025-05-07 15:14:03.903 | DEBUG    | __main__:coarsen_healpix_region:452 - (8320, 8384)
133
2025-05-07 15:14:53.955 | DEBUG    | __main__:coarsen_healpix_region:452 - (8512, 8576)
[8328 8569]
Confirming idx for 5
130
2025-05-07 15:17:02.074 | DEBUG    | __main__:coarsen_healpix_region:452 - (8320, 8384)
133
2

## Move affected 3D donefiles

In [71]:
ds = cat['um_glm_n2560_RAL3p3'](zoom=10, time='PT3H').to_dask()

/home/users/mmuetz/miniforge3/envs/hackathon_env/lib/python3.12/site-packages/intake_xarray/base.py:21: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  'dims': dict(self._ds.dims),


In [72]:
time = ds.time.values
tmask = (time >= pd.Timestamp('2021-01-01 00:00')) & (time <= pd.Timestamp('2021-01-11 01:00'))

In [73]:
affected_indices = np.where(tmask)[0]
affected_indices

array([2776, 2777, 2778, 2779, 2780, 2781, 2782, 2783, 2784, 2785, 2786,
       2787, 2788, 2789, 2790, 2791, 2792, 2793, 2794, 2795, 2796, 2797,
       2798, 2799, 2800, 2801, 2802, 2803, 2804, 2805, 2806, 2807, 2808,
       2809, 2810, 2811, 2812, 2813, 2814, 2815, 2816, 2817, 2818, 2819,
       2820, 2821, 2822, 2823, 2824, 2825, 2826, 2827, 2828, 2829, 2830,
       2831, 2832, 2833, 2834, 2835, 2836, 2837, 2838, 2839, 2840, 2841,
       2842, 2843, 2844, 2845, 2846, 2847, 2848, 2849, 2850, 2851, 2852,
       2853, 2854, 2855, 2856])

In [74]:
z_affected_idx = {}
for z in range(9, -1, -1):
    print(z)
    zchunk = config['groups']['3d']['chunks'][z][0]
    nchunk = int(np.ceil(len(tmask) / zchunk))
    ztime_idx = np.arange(nchunk).repeat(zchunk)
    z_affected_idx[z] = np.unique(ztime_idx[affected_indices])

9
8
7
6
5
4
3
2
1
0


In [75]:
z_affected_idx

{9: array([694, 695, 696, 697, 698, 699, 700, 701, 702, 703, 704, 705, 706,
        707, 708, 709, 710, 711, 712, 713, 714]),
 8: array([173, 174, 175, 176, 177, 178]),
 7: array([43, 44]),
 6: array([43, 44]),
 5: array([43, 44]),
 4: array([10, 11]),
 3: array([2]),
 2: array([0]),
 1: array([0]),
 0: array([0])}

In [76]:
for z in range(9, -1, -1):
    z_idx = z_affected_idx[z]
    print(affected_indices[[0, -1]])
    print(f'Confirming idx for {z}')
    print(z_idx[0])
    print(Path(f'/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z{z}/{z_idx[0]}.done').read_text().split('\n')[0])
    print(z_idx[-1])
    print(Path(f'/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z{z}/{z_idx[-1]}.done').read_text().split('\n')[0])    

[2776 2856]
Confirming idx for 9
694
2025-05-07 09:25:59.775 | DEBUG    | __main__:coarsen_healpix_region:452 - (2776, 2780)
714
2025-05-07 09:47:39.992 | DEBUG    | __main__:coarsen_healpix_region:452 - (2856, 2860)
[2776 2856]
Confirming idx for 8
173
2025-05-04 21:59:13.932 | DEBUG    | __main__:coarsen_healpix_region:447 - (2768, 2784)
178
2025-05-04 21:43:32.479 | DEBUG    | __main__:coarsen_healpix_region:447 - (2848, 2864)
[2776 2856]
Confirming idx for 7
43
2025-05-04 23:00:33.126 | DEBUG    | __main__:coarsen_healpix_region:453 - (2752, 2816)
44
2025-05-04 23:09:48.258 | DEBUG    | __main__:coarsen_healpix_region:453 - (2816, 2880)
[2776 2856]
Confirming idx for 6
43
2025-05-04 23:53:20.605 | DEBUG    | __main__:coarsen_healpix_region:453 - (2752, 2816)
44
2025-05-04 23:56:28.924 | DEBUG    | __main__:coarsen_healpix_region:453 - (2816, 2880)
[2776 2856]
Confirming idx for 5
43
2025-05-05 00:04:48.782 | DEBUG    | __main__:coarsen_healpix_region:453 - (2752, 2816)
44
2025-05-0

**Conclusion:** correctly generating index of file at each lower zoom level (affected indices 2776-2856 are always contained in the first and last at each zoom)

In [84]:
for z in range(9, -1, -1):
    z_idx = z_affected_idx[z]
    print(f'zoom {z}')
    for idx in z_idx:
        path = Path(f'/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z{z}/{idx}.done')
        assert path.exists()
        newpath = path.with_suffix('.jan_data_issue')
        print(f'  {path} -> {newpath.name}')
        newpath = path.rename(newpath)


zoom 9
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z9/694.done -> 694.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z9/695.done -> 695.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z9/696.done -> 696.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z9/697.done -> 697.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z9/698.done -> 698.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z9/699.done -> 699.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z9/700.done -> 700.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z9/701.done -> 701.jan_data_issue
  /gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z9/702.done -> 702.jan_data_issue
  /gws/nopw/j04/hrcm/

In [85]:
for z in range(9, -1, -1):
    z_idx = z_affected_idx[z]
    print(affected_indices[[0, -1]])
    print(f'Confirming idx for {z}')
    print(z_idx[0])
    print(Path(f'/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z{z}/{z_idx[0]}.jan_data_issue').read_text().split('\n')[0])
    print(z_idx[-1])
    print(Path(f'/gws/nopw/j04/hrcm/mmuetz/slurm_done/dev/glm.n2560_RAL3p3/v5/coarsen/3d/z{z}/{z_idx[-1]}.jan_data_issue').read_text().split('\n')[0])    

[2776 2856]
Confirming idx for 9
694
2025-05-07 09:25:59.775 | DEBUG    | __main__:coarsen_healpix_region:452 - (2776, 2780)
714
2025-05-07 09:47:39.992 | DEBUG    | __main__:coarsen_healpix_region:452 - (2856, 2860)
[2776 2856]
Confirming idx for 8
173
2025-05-04 21:59:13.932 | DEBUG    | __main__:coarsen_healpix_region:447 - (2768, 2784)
178
2025-05-04 21:43:32.479 | DEBUG    | __main__:coarsen_healpix_region:447 - (2848, 2864)
[2776 2856]
Confirming idx for 7
43
2025-05-04 23:00:33.126 | DEBUG    | __main__:coarsen_healpix_region:453 - (2752, 2816)
44
2025-05-04 23:09:48.258 | DEBUG    | __main__:coarsen_healpix_region:453 - (2816, 2880)
[2776 2856]
Confirming idx for 6
43
2025-05-04 23:53:20.605 | DEBUG    | __main__:coarsen_healpix_region:453 - (2752, 2816)
44
2025-05-04 23:56:28.924 | DEBUG    | __main__:coarsen_healpix_region:453 - (2816, 2880)
[2776 2856]
Confirming idx for 5
43
2025-05-05 00:04:48.782 | DEBUG    | __main__:coarsen_healpix_region:453 - (2752, 2816)
44
2025-05-0